# Verification of the growth law — what the checks establish, and what they do not

An executable version of [`DEVIATOR_SCALING_FINDING.md`](../DEVIATOR_SCALING_FINDING.md)
and [`VISCOUS_UPDATE_SCHEME.md`](../VISCOUS_UPDATE_SCHEME.md). Every number below is
computed here from the Fortran under test, not quoted.

The reason to have this as a notebook rather than only as prose: the central claim is
that **agreement between two implementations of the same expression is not correctness**,
and that is far more convincing when the reader can change `alpha` and watch it.

Three things are shown, in order:

1. the implementation matches an *independent* closed form up to a term we can predict;
2. that term is **purely spherical**, so von Mises — what the thesis reports — is untouched;
3. the viscous step limit, and the guard that now enforces it.

**Requires** `gfortran` and a C compiler. Run from the repository root or from
`ansys_usermat/`.

In [1]:
import subprocess, sys, tempfile, shutil
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "ansys_usermat").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
AU = ROOT / "ansys_usermat"
sys.path.insert(0, str(AU / "apdl"))
import closed_form_reference as cf

FC = shutil.which("gfortran"); CC = shutil.which("cc") or shutil.which("gcc")
assert FC and CC, "needs gfortran and a C compiler"
print("repository root:", ROOT)

repository root: /home/user/pde-fem-biofilm


## Build the drivers

Two: the **core** (no guard — this is the routine that is 0 ULP against the Abaqus UMAT)
and the **wrapper** we hand to the partner group, which adds the step guard.

In [2]:
TMP = Path(tempfile.mkdtemp())
def sh(*a): subprocess.run([str(x) for x in a], check=True, cwd=TMP)

sh(FC,"-c","-ffixed-line-length-132","-J",TMP, AU/"coupling/usermat_py_hook.f","-o",TMP/"hook.o")
sh(FC,"-c","-ffixed-line-length-132","-I",TMP, AU/"usermat_biofilm.f","-o",TMP/"core.o")
sh(CC,"-c","-fPIC", AU/"coupling/biofilm_py_eval.c","-o",TMP/"shim.o")
sh(FC,"-ffixed-line-length-132","-I",TMP, AU/"crosscheck/xcheck_driver_ans.f",
   TMP/"core.o",TMP/"hook.o",TMP/"shim.o","-o",TMP/"core")
sh(FC,"-ffixed-line-length-132","-I",TMP, AU/"crosscheck/wrapper_driver.f",
   AU/"biofilm_material_v01.f", TMP/"core.o",TMP/"hook.o",TMP/"shim.o","-o",TMP/"wrap")

I3 = np.eye(3)
ENV = {"PATH": "/usr/bin:/bin"}

def run_core(F, Fv, alpha, c10, c01, d1, eta, mtype, dt):
    txt = (" ".join(f"{F[i,j]:.17e}" for i in range(3) for j in range(3)) + "\n" +
           " ".join(f"{Fv[i,j]:.17e}" for i in range(3) for j in range(3)) + "\n" +
           f"{alpha:.17e} {c10:.17e} {c01:.17e} {d1:.17e} {eta:.17e} {mtype:.1f} {dt:.17e}\n")
    out = subprocess.run([str(TMP/"core")], input=txt, capture_output=True,
                         text=True, env=ENV, timeout=30)
    assert out.returncode == 0, out.stderr
    v = [float(x) for x in out.stdout.split()]
    return np.array(v[:6]), np.array(v[6:15]).reshape(3,3)

def mises(s):
    a,b,c,d,e,f = s
    return np.sqrt(0.5*((a-b)**2+(b-c)**2+(c-a)**2) + 3*(d*d+e*e+f*f))

print("built")

built


## 1. Against an independent closed form

`closed_form_reference.py` derives both growth cases from the continuum statement and
imports nothing from the code under test. This matters: the reference values shipped with
the verification decks are produced by *calling* the implementation, so agreeing with them
establishes nothing.

For fully constrained isotropic growth (`F = I`) the elastic deformation is isotropic, so
the deviator of any potential with an isochoric/volumetric split vanishes and only the
volumetric term survives — independently of the shear constants **and of the viscosity**,
since the deviatoric flow driver is zero too.

In [3]:
C10, C01, D1, MTYPE, DT = 2.0e-4, 0.0, 5000.0, 0.0, 5.0

print(f"{'alpha':>6} {'closed form':>15} {'+ predicted':>15} {'implementation':>16} {'rel.err':>10}")
for a in (0.02, 0.05, 0.10, 0.20, 0.35):
    impl, _  = run_core(I3, I3, a, C10, C01, D1, 0.0, MTYPE, DT)
    exact    = cf.constrained_stress(a, D1)
    predicted = exact + cf.spurious_term(a, C10)
    rel = abs(impl[0]-predicted[0])/abs(predicted[0])
    print(f"{a:6.2f} {exact[0]:15.6e} {predicted[0]:15.6e} {impl[0]:16.6e} {rel:10.1e}")

 alpha     closed form     + predicted   implementation    rel.err
  0.02   -2.307107e-05   -4.022019e-05    -4.022019e-05    8.4e-16
  0.05   -5.446496e-05   -1.019276e-04    -1.019276e-04    1.3e-16
  0.10   -9.947408e-05   -2.112781e-04    -2.112781e-04    3.8e-16
  0.20   -1.685185e-04   -4.726465e-04    -4.726465e-04    5.7e-16
  0.35   -2.374232e-04   -1.046887e-03    -1.046887e-03    2.1e-16


The implementation does **not** match the closed form — and the gap is not noise. It is
reproduced to ~1e-16 by a term we can write down:

$$\sigma_{\text{spurious}} = \frac{2\,C_{10}}{J_e}\left[1-(1+\alpha)^2\right]\mathbf I$$

which comes from the isochoric split applying $J^{-2/3}$ to the subtracted trace but not to
the tensor beside it. The two forms agree exactly at $J=1$ and nowhere else — and growth is
what moves $J_e$ away from 1.

So this is not a case of the checks being loose. **They cannot see it**, and the next cell
shows why.

## 2. The error is purely spherical

This is the bound that decides how much the finding actually matters. Compare against a
correctly-split reference over *general, non-isotropic* deformations.

In [4]:
def sigma_ref(F, a, c10, c01, d1, mt):
    """Correct isochoric split, written independently of the Fortran."""
    Fe = F @ (I3/(1.0+a)); J = np.linalg.det(Fe); b = Fe @ Fe.T
    bb = J**(-2.0/3.0) * b
    tau = 2.0*c10*(bb - np.trace(bb)/3.0*I3)
    if mt > 0.5:
        T = np.trace(bb)*bb - bb@bb
        tau = tau + 2.0*c01*(T - np.trace(T)/3.0*I3)
    sig = tau/J + (2.0/d1)*(J-1.0)*I3
    return np.array([sig[0,0],sig[1,1],sig[2,2],sig[0,1],sig[1,2],sig[0,2]])

Fs = {"shear + stretch": np.array([[1.06,0.02,0.0],[0.0,0.97,0.01],[0.0,0.0,0.98]]),
      "strong shear":    np.array([[1.10,0.12,0.03],[0.05,0.92,0.00],[0.02,0.04,1.05]]),
      "F = I":           I3}

print(f"{'state':>16} {'alpha':>6} {'dev. part of diff':>19} {'shear part':>12} {'d(von Mises)':>14}")
for name, F in Fs.items():
    for a in (0.05, 0.20):
        impl,_ = run_core(F, I3, a, C10, C01, D1, 0.0, MTYPE, DT)
        ref    = sigma_ref(F, a, C10, C01, D1, MTYPE)
        d = impl - ref
        print(f"{name:>16} {a:6.2f} {np.max(np.abs(d[:3]-d[:3].mean())):19.2e} "
              f"{np.max(np.abs(d[3:])):12.2e} {abs(mises(impl)-mises(ref)):14.2e}")

           state  alpha   dev. part of diff   shear part   d(von Mises)
 shear + stretch   0.05            1.36e-20     1.69e-21       1.36e-20
 shear + stretch   0.20            5.42e-20     0.00e+00       4.07e-20
    strong shear   0.05            3.39e-20     0.00e+00       0.00e+00
    strong shear   0.20            5.42e-20     0.00e+00       0.00e+00
           F = I   0.05            0.00e+00     0.00e+00       0.00e+00
           F = I   0.20            0.00e+00     0.00e+00       0.00e+00


Zero deviatoric part, zero shear part, zero change in von Mises — at machine precision,
for every state including strongly non-isotropic ones. The difference is a pure pressure
error.

**So the reported quantity is untouched at fixed $F$.** That also explains, rather than
contradicts, the dual-UMAT growth cross-check that agrees to 0.1%: it compares von Mises,
which cannot see a spherical term. That check is blind to this by construction, not by
accident.

What is *not* settled: in a real solve a wrong pressure changes equilibrium, hence
displacements, hence von Mises indirectly. That needs a run to measure.

## 3. The viscous step limit, and the guard

The flow increment is evaluated at the *old* viscous state, so the update is explicit —
whatever the label "backward Euler" elsewhere in the repository suggests. It therefore has
a step restriction, which the sweep below makes concrete.

In [5]:
c10_stiff = 0.5*(1000.0/(2*1.30))/1.15     # the wrapper's (E,nu) -> C10 map at E=1000
eta = 5.0
tau = eta/(2*c10_stiff)
print(f"C10 = {c10_stiff:.1f},  eta = {eta},  relaxation time tau = {tau:.4f} s\n")

print(f"{'dt':>10} {'dt/tau':>8} {'sigma_11':>14}   note")
for dt in (1e-4, 1e-3, 3e-3, 5e-3, 7.5e-3, 1e-2, 2e-2):
    s,_ = run_core(Fs["shear + stretch"], I3, 0.2, c10_stiff, 0.15*c10_stiff,
                   2.0/(1000.0/(3*(1-2*0.30))), eta, 1.0, dt)
    note = ""
    if dt/tau > 1.0:  note = "diverging"
    elif s[0] > 0:    note = "SIGN FLIPPED"
    print(f"{dt:10.1e} {dt/tau:8.3f} {s[0]:14.4f}   {note}")

C10 = 167.2,  eta = 5.0,  relaxation time tau = 0.0150 s

        dt   dt/tau       sigma_11   note
   1.0e-04    0.007      -513.1331   
   1.0e-03    0.067      -442.3254   
   3.0e-03    0.201      -289.2323   
   5.0e-03    0.334      -133.7931   
   7.5e-03    0.502        81.4774   SIGN FLIPPED
   1.0e-02    0.669       347.0955   SIGN FLIPPED
   2.0e-02    1.338      3287.5958   diverging


The stress crosses zero near `dt/tau ~ 0.5` and runs away past 1 — not something an
unconditionally stable scheme does.

Because the *caller* sets `dt`, leaving this to documentation would mean a wrong step
returns a plausible-looking wrong stress. The delivered wrapper therefore refuses it:

In [6]:
def run_wrap(dt, eta=5.0, biofilm=1.0):
    F = Fs["shear + stretch"]
    txt = (" ".join(f"{F[i,j]:.17e}" for i in range(3) for j in range(3)) + "\n" +
           " ".join(f"{I3[i,j]:.17e}" for i in range(3) for j in range(3)) + "\n" +
           f"{1000.0:.17e} {1.0:.17e} {0.30:.17e} {0.30:.17e} {biofilm:.17e} "
           f"{0.2:.17e} {eta:.17e} {dt:.17e} {0.15:.17e} 1.0\n")
    out = subprocess.run([str(TMP/"wrap")], input=txt, capture_output=True,
                         text=True, env=ENV, timeout=30)
    v = [float(x) for x in out.stdout.split()]
    return int(v[15]), np.array(v[0:6])

print(f"{'dt/tau':>8} {'sKeyCut':>8}   outcome")
for r in (0.007, 0.1, 0.4, 0.6, 1.0, 5.0):
    kc, s = run_wrap(r*tau)
    print(f"{r:8.3f} {kc:8d}   " + ("refused -> solver cuts back"
                                    if kc else f"answered, sigma_11 = {s[0]:.2f}"))

print("\nelastic run (eta = 0): no relaxation time, so never restricted")
for dt in (1e-4, 1.0, 1e6):
    kc, _ = run_wrap(dt, eta=0.0)
    print(f"  dt = {dt:8.0e}  sKeyCut = {kc}")

  dt/tau  sKeyCut   outcome
   0.007        0   answered, sigma_11 = -512.76
   0.100        0   answered, sigma_11 = -404.10
   0.400        0   answered, sigma_11 = -53.39
   0.600        1   refused -> solver cuts back
   1.000        1   refused -> solver cuts back
   5.000        1   refused -> solver cuts back

elastic run (eta = 0): no relaxation time, so never restricted
  dt =    1e-04  sKeyCut = 0
  dt =    1e+00  sKeyCut = 0
  dt =    1e+06  sKeyCut = 0


**Read that table carefully: "answered" does not mean "accurate."** At `dt/tau = 0.4`
the step is accepted and returns −53 Pa where a resolved step gives −513 — the sign is
right and the magnitude is not.

That is deliberate, and it is the honest way round. The guard exists to stop the
*catastrophic* failure, where the answer is qualitatively wrong and nothing says so. How
much accuracy to buy with step size below that is a convergence decision the caller has to
make with their own tolerances, and a routine that refused every inaccurate step would
stall solves that its user had legitimately chosen to run coarsely.

So the guard is a floor, not a certificate. `DTMAX_RATIO` in
`biofilm_material_v01.f` is where to raise the floor if a particular study needs it.


## What to take from this

- The 0-ULP cross-implementation result is real and worth reporting — but it establishes
  that two ports compute the same expression identically, **not** that the expression is
  right. Section 1 is the demonstration.
- The discrepancy that exists is bounded and characterised: a pure pressure error, with
  von Mises unaffected at fixed $F$. It is documented rather than corrected, for the
  reasons in `DEVIATOR_SCALING_FINDING.md` §7.
- The step restriction is a property of the integrator, and the routine we hand over
  enforces it rather than trusting the caller to have read about it.

Reproduced non-interactively by `tests/test_closed_form_reference.py`,
`tests/test_material_wrapper.py` and
`ansys_usermat/crosscheck/check_deviator_scaling.py`.